# Script for parsing subtitles

In [55]:
# imports

import numpy
from stl import mesh
import pandas as pd
import numpy as np
import os
import ttconv
import subprocess
from pathlib import Path
import re
from tqdm import tqdm

# step 1: converting stl to rtf files

In [ ]:
p = r"/Users/tomschulz/Documents/GitHub/HUMAN/subtitles/HUMAN_DATASCHOOL/HUMAN_DATASCHOOL"
output_path = r"/Users/tomschulz/Documents/GitHub/HUMAN/subtitles/CONVERTED_SUBTITLES" 

for e in os.scandir(p):
    if e.is_file():
        with open(e.path, "r") as f:
            print("Content of", e.name, ":")
            # print(f.read())
            output_name = e.name.replace(".stl", ".srt")

            # tt convert -i <input_file> -o <output_file> for the file
            subprocess.run(
                ["tt", "convert", "-i", str(e.path), "-o", str(os.path.join(output_path, output_name))],
                check=True
            )

            print("Conversion done.")

# check to find missing files

In [79]:


stl_path = "/Users/tomschulz/Documents/GitHub/HUMAN/subtitles/HUMAN_DATASCHOOL/HUMAN_DATASCHOOL"
srt_path = "/Users/tomschulz/Documents/GitHub/HUMAN/subtitles/CONVERTED_SUBTITLES"

# Get filenames without extensions
stl_files = {os.path.splitext(f.name)[0] for f in os.scandir(stl_path) if f.is_file() and f.name.endswith(".stl")}
srt_files = {os.path.splitext(f.name)[0] for f in os.scandir(srt_path) if f.is_file() and f.name.endswith(".srt")}

# Files present in STL but missing in SRT
missing = stl_files - srt_files

print(f"{len(missing)} files missing conversion:")
for f in sorted(missing):
    print(f)

2 files missing conversion:
2015-05-29-22,57-1
2015-09-17-08,10-1


# converting also the missing files

In [78]:
# --- Paths ---
stl_path = Path("/Users/tomschulz/Documents/GitHub/HUMAN/subtitles/HUMAN_DATASCHOOL/HUMAN_DATASCHOOL")
srt_path = Path("/Users/tomschulz/Documents/GitHub/HUMAN/subtitles/CONVERTED_SUBTITLES")



# --- Convert only missing files ---
for name in tqdm(missing, desc="Converting STL → SRT"):
    input_file = stl_path / f"{name}.stl"
    output_file = srt_path / f"{name}.srt"

    try:
        subprocess.run(
            ["tt", "convert", "-i", str(input_file), "-o", str(output_file)],
            check=True,
            capture_output=True,
            text=True
        )
    except subprocess.CalledProcessError as e:
        print(f"\n❌ Failed for {name}")
        print(e.stderr)

print("Done.")

Converting STL → SRT: 100%|██████████| 2/2 [00:00<00:00,  8.58it/s]


❌ Failed for 2015-05-29-22,57-1
Input file is /Users/tomschulz/Documents/GitHub/HUMAN/subtitles/HUMAN_DATASCHOOL/HUMAN_DATASCHOOL/2015-05-29-22,57-1.stl
Output file is /Users/tomschulz/Documents/GitHub/HUMAN/subtitles/CONVERTED_SUBTITLES/2015-05-29-22,57-1.srt
Traceback (most recent call last):
  File "/Users/tomschulz/Documents/GitHub/HUMAN/myenv/bin/tt", line 6, in <module>
    sys.exit(main())
  File "/Users/tomschulz/Documents/GitHub/HUMAN/myenv/lib/python3.9/site-packages/ttconv/tt.py", line 482, in main
    args.func(args)
  File "/Users/tomschulz/Documents/GitHub/HUMAN/myenv/lib/python3.9/site-packages/ttconv/tt.py", line 335, in convert
    model = stl_reader.to_model(f, reader_config, progress_callback_read)
  File "/Users/tomschulz/Documents/GitHub/HUMAN/myenv/lib/python3.9/site-packages/ttconv/stl/reader.py", line 68, in to_model
    progress_callback(i/m.get_tti_count())
ZeroDivisionError: division by zero


❌ Failed for 2015-09-17-08,10-1
Input file is /Users/tomschulz/Do

In [213]:
from striprtf.striprtf import rtf_to_text

def rtf_to_plain_text(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return rtf_to_text(f.read())

# part 3: stl to df

In [ ]:
import pandas as pd
import re

def srt_to_df(path: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        content = f.read().strip()

    content = content.replace("\r\n", "\n").replace("\r", "\n")
    blocks = re.split(r"\n\s*\n", content)

    rows = []
    for block in blocks:
        lines = block.split("\n")
        if len(lines) < 3:
            continue

        try:
            idx = int(lines[0].strip())
        except ValueError:
            continue

        m = re.match(r"(.+?)\s*-->\s*(.+)", lines[1].strip())
        if not m:
            continue

        start, end = m.groups()
        text = " ".join(line.strip() for line in lines[2:] if line.strip())

        rows.append({"index": idx, "start": start.strip(), "end": end.strip(), "text": text})

    # Force expected columns even if empty
    return pd.DataFrame(rows, columns=["index", "start", "end", "text"])

In [ ]:

path = r"/Users/tomschulz/Documents/GitHub/HUMAN/subtitles/CONVERTED_SUBTITLES"

rows = []
files = [f for f in os.scandir(path) if f.is_file() and f.name.endswith(".srt")]

for f in tqdm(files, desc="Processing subtitles"):
    df = srt_to_df(f.path)

    if df.empty:
        # Useful debugging: show first couple lines to see what's wrong with that file
        with open(f.path, "r", encoding="utf-8", errors="replace") as fh:
            preview = "".join([next(fh, "") for _ in range(5)])
        tqdm.write(f"Skipped (no parsed subtitles): {f.name}\nPreview:\n{preview}")
        continue

    full_text = "\n".join(df["text"].astype(str))
    rows.append({"programma": f.name, "text": full_text})

combined = pd.DataFrame(rows)
combined

# part 4: cleaning df

In [83]:
# cleaning

import re
import html
import pandas as pd

BOILERPLATE_PATTERNS = [
    r"^\s*LIVEPROGRAMMA.*$",
    r"^\s*ONDERTITELING KAN ACHTERLOPEN\s*$",
    r"^\s*DIT PROGRAMMA WERD LIVE ONDERTITELD\s*$",
    r"^\s*NPO\s*ONDERTITELING.*$",
    r"^\s*TT888.*$",
    r"^\s*informatie:\s*.*$",
]

# Lines that are often just non-dialogue cues; remove if the line is ONLY this cue (optionally repeated)
CUE_ONLY = r"^(APPLAUS|GELACH|MUZIEK.*|ENGELS|BROM(\s+BROM)*)$"

TAG_RE = re.compile(r"<[^>]+>")
MULTISPACE_RE = re.compile(r"[ \t]+")
MULTINE_RE = re.compile(r"\n{3,}")

def clean_subtitle_text(text: str) -> str:
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""

    # Normalize newlines
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Unescape HTML entities (&amp; etc.)
    text = html.unescape(text)

    # Remove HTML-like tags (<font ...>, </font>, etc.)
    text = TAG_RE.sub("", text)

    # Remove bracketed cues like [APPLAUS] or (GELACH)
    text = re.sub(r"\[(.*?)\]", r"\1", text)   # keep inside text, drop brackets
    text = re.sub(r"\((.*?)\)", r"\1", text)   # keep inside text, drop parentheses

    # Remove boilerplate lines
    lines = []
    for line in text.split("\n"):
        stripped = line.strip()
        if not stripped:
            continue

        # Drop known boilerplate patterns
        if any(re.match(p, stripped, flags=re.IGNORECASE) for p in BOILERPLATE_PATTERNS):
            continue

        # Drop cue-only lines
        if re.match(CUE_ONLY, stripped, flags=re.IGNORECASE):
            continue

        lines.append(stripped)

    # Re-join lines
    cleaned = "\n".join(lines)

    # Fix spacing before punctuation (common after tag removal / joins)
    cleaned = re.sub(r"\s+([.,!?;:])", r"\1", cleaned)

    # Collapse extra spaces/tabs
    cleaned = MULTISPACE_RE.sub(" ", cleaned)

    # Collapse excessive blank lines
    cleaned = MULTINE_RE.sub("\n\n", cleaned)

    return cleaned.strip()

In [131]:
combined["text_clean"] = combined["text"].apply(clean_subtitle_text)

In [ ]:
# drop random sports episode
combined = combined.drop(index=9880)

In [ ]:
# delete column
combined.drop("text", axis = 1, inplace=True)



In [ ]:
# safe ro csv
combined.to_csv("/Users/tomschulz/Documents/GitHub/HUMAN/subtitles/parsed_and_cleaned_subtitles.csv")

## read additional info files

In [223]:
xls = pd.ExcelFile("/Users/tomschulz/Documents/GitHub/HUMAN/subtitles/File_Index_HUNAN-13-01-2026.xlsx")
df1 = pd.read_excel(xls, 'SIEN')
df2 = pd.read_excel(xls, 'OTTO')

In [292]:
df = combined

In [266]:
df['programma'] = df['programma'].str.replace('.stl', '', regex=False)

In [293]:
df['programma'].iloc[0]

'2017-11-27-19,01-1'

In [296]:
df['year'] = df['STL filename_y'].str.extract(r'^(\d{4})')

In [294]:
df = df.merge(
    df1[['STL filename', 'date_publication',  'program_title']],
    how='left',
    left_on='programma',
    right_on='STL filename'
)

df = df.rename(columns={'date_publication': 'date1'})
# df = df.drop(columns=['STL filename'])

In [295]:
df = df.merge(
    df2[['programmatitel', 'STL filename']],
    how='left',
    left_on='programma',
    right_on='STL filename'
)

df = df.rename(columns={'date_publication': 'date2'})
# df = df.drop(columns=['STL filename'])

In [297]:
# Alleen vullen waar year nog NaN is
mask = df['year'].isna()
df.loc[mask, 'year'] = pd.to_datetime(df.loc[mask, 'date1'], errors='coerce').dt.year

# Daarna fallback naar STL filename_x (ook alleen waar year nog NaN is)
mask = df['year'].isna()
df.loc[mask, 'year'] = df.loc[mask, 'STL filename_x'].str[:4]

In [315]:
df.drop('STL filename_y', axis=1, inplace=True)

In [312]:
df['filename'] = df['STL filename_x'].combine_first(df['STL filename_y'])

In [317]:
df = df[['filename', 'program', 'year', 'text_clean']]

In [320]:
df.to_csv('parsed_and_cleaned_subtitles.csv')

In [318]:
df

filename                program  year  \
0            2017-11-27-19,01-1  DE WERELD DRAAIT DOOR  2017   
1     GOEDEMORGEN_N-WON02205666  Goedemorgen Nederland  2021   
2     GOEDEMORGEN_N-WON02094879  Goedemorgen Nederland  2020   
3            2017-03-17-08,11-1  GOEDEMORGEN NEDERLAND  2017   
4     GOEDEMORGEN_N-WON02199391  Goedemorgen Nederland  2021   
...                         ...                    ...   ...   
9995         2016-01-27-07,40-1  GOEDEMORGEN NEDERLAND  2016   
9996  GOEDEMORGEN_N-WON02386562  Goedemorgen Nederland  2023   
9997         2015-11-09-07,40-1  GOEDEMORGEN NEDERLAND  2015   
9998         2017-02-17-07,11-1  GOEDEMORGEN NEDERLAND  2017   
9999  DE_WERELD_DRA-WON02080353  De Wereld Draait Door  2020   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      